# GEO-Bench-VLM — LLaVA-OneVision-7B Evaluation

**Required GPU:** A100 (40 GB) recommended — OOM likely on T4/L4 for 7B  
**Note:** This notebook pins `transformers==4.45.2` (required by LLaVA-NeXT). Do **not** mix this session with Qwen or LFM notebooks.  
**Prerequisite:** Run `notebooks/download_dataset.ipynb` once to download the dataset to Drive.  
Results are saved to Google Drive. Run cells top-to-bottom.

In [4]:
# ── CONFIG — only edit this cell ─────────────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/GEOBench'  # root folder on your Drive
REPO_URL     = 'https://github.com/Tarekbouamer/GEOBench-VLM'
MAX_SAMPLES  = None   # int (e.g. 10) for a smoke test, None for full benchmark
RESULTS_DIR  = f'{DRIVE_BASE}/results'             # where eval JSON results are saved


In [5]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA GeForce RTX 3090, 24576 MiB


In [6]:
# ── 2. Mount Drive & configure paths ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

WEIGHTS_DIR = f'{DRIVE_BASE}/Out_weights'
DATA_PATH   = f'{DRIVE_BASE}/GEOBench-VLM'
REPO        = '/content/GEO-Bench-VLM'

for d in [WEIGHTS_DIR, DATA_PATH, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ['DATA_PATH']                = DATA_PATH
os.environ['WEIGHTS_DIR']             = WEIGHTS_DIR
os.environ['RESULTS_DIR']             = RESULTS_DIR
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'DATA_PATH   = {DATA_PATH}')
print(f'WEIGHTS_DIR = {WEIGHTS_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# ── 3. Clone repo ─────────────────────────────────────────────────────────────
import os
REPO = '/content/GEO-Bench-VLM'
if not os.path.isdir(REPO):
    !git clone {REPO_URL} {REPO}

weights_link = f'{REPO}/Out_weights'
if not os.path.exists(weights_link):
    os.symlink(os.environ['WEIGHTS_DIR'], weights_link)

%cd {REPO}
print('Repo ready.')

In [ ]:
# ── 4. HuggingFace login ──────────────────────────────────────────────────────
# Add HF_TOKEN to Colab Secrets (key icon in sidebar) for unattended runs.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via Colab secret HF_TOKEN.')
except Exception:
    from huggingface_hub import login
    login()  # fallback: interactive prompt

In [ ]:
# ── 5. Verify dataset is present ─────────────────────────────────────────────
# Run notebooks/download_dataset.ipynb FIRST if you haven't already.
import os

dataset_dir = os.environ['DATA_PATH']
single_dir  = os.path.join(dataset_dir, 'Single')

if not os.path.isdir(single_dir):
    raise FileNotFoundError(
        f"Dataset not found at {dataset_dir}\n"
        "Please run notebooks/download_dataset.ipynb first."
    )

splits = ['Single', 'Temporal', 'Captioning', 'Ref-Det', 'Ref-Seg']
for split in splits:
    qa = os.path.join(dataset_dir, split, 'qa.json')
    status = '✓' if os.path.isfile(qa) else '✗ missing'
    print(f'  {split:<12} {status}')
print('Dataset ready.')

In [ ]:
# ── 6. Clone LLaVA-NeXT & install dependencies ───────────────────────────────
# LLaVA-NeXT requires transformers==4.45.2 (apply_chunking_to_forward
# was removed in 4.46+). Install it LAST to avoid it getting overridden.
import os, shutil
REPO = '/content/GEO-Bench-VLM'
LLAVA_NEXT = f'{REPO}/LLaVA-NeXT'

# Check for setup.py, not just the directory — a partial/empty clone won't have it
if not os.path.isfile(f'{LLAVA_NEXT}/setup.py'):
    if os.path.isdir(LLAVA_NEXT):
        shutil.rmtree(LLAVA_NEXT)   # remove incomplete clone
    !git clone https://github.com/LLaVA-VL/LLaVA-NeXT {LLAVA_NEXT}

!pip install -q -e {LLAVA_NEXT}/.[train]
!pip install -q "transformers==4.45.2"   # pin AFTER llava install
# LLaVA-NeXT downgrades huggingface_hub — restore a version that has HF_HUB_ENABLE_HF_TRANSFER
!pip install -q --upgrade huggingface_hub
print('Dependencies installed.')


In [ ]:
# ── 7. Download model weights (idempotent) ────────────────────────────────────
import importlib, os
import huggingface_hub
importlib.reload(huggingface_hub)          # reload after pip upgrade in cell 6
from huggingface_hub import snapshot_download

model_dir = os.path.join(os.environ['WEIGHTS_DIR'], 'llava-onevision-qwen2-7b-si')
if os.path.isdir(model_dir):
    print('Weights already present — skipping download.')
else:
    print('Downloading llava-onevision-qwen2-7b-si weights...')
    snapshot_download('lmms-lab/llava-onevision-qwen2-7b-si', local_dir=model_dir)
    print('Weights downloaded.')


In [ ]:
# ── 8. Run evaluation ─────────────────────────────────────────────────────────
import sys, os

REPO = '/content/GEO-Bench-VLM'
sys.path.insert(0, f'{REPO}/eval_geobenchvlm')
from llavaone1_cls_single import main

main(os.environ['DATA_PATH'], os.environ['RESULTS_DIR'], MAX_SAMPLES or None)
print('Evaluation complete.')


In [ ]:
# ── 9. Confirm results saved to Drive ────────────────────────────────────────
import glob, os, json

results_folder = os.path.join(os.environ['RESULTS_DIR'], 'llava-onevision-7b')
json_files = glob.glob(os.path.join(results_folder, '*.json'))

if not json_files:
    print('No result files found — did the eval complete?')
else:
    print(f'Results saved at: {results_folder}')
    for f in json_files:
        with open(f) as fh:
            data = json.load(fh)
        print(f'  {os.path.basename(f)} — {len(data)} entries')
    print('\nRun notebooks/score_results.ipynb to compare models.')
